# Ablation: Generator Static Attributes as Output-Head Conditioning

Follow-up to `baseline_transformer.ipynb`'s Transformer encoder baseline ("v1", ~0.71 validation
micro-average ROC AUC). v1's output head is a single shared `Linear(d_model, 14)` layer — "generator
identity" is purely "which of 14 output columns", with no physical information about the generator
attached to it.

This notebook tests whether conditioning the output head on each generator's static physical attributes
— `p_min`, `p_max`, `p_nom`, `startcost_const`, `stopcost_const`, `penstock`, from
`data/extended/Tokke_Vinje_topology.yaml` — helps, especially for small/unusual units like `Haukeli_G1`
(a tiny 0.3-4.9 MW peaking unit) that the per-generator breakdown in the baseline notebook showed were
predicted worst.

Kept in its own notebook rather than folded into the baseline so the main baseline stays simple. v1 is
retrained here (rather than hardcoding its reported score) so the comparison is apples-to-apples against
this run, using identical data/split/training code to `baseline_transformer.ipynb`.

**Result (spoiler): this did not help overall** — see the conclusion at the end.

## 1. Setup

In [1]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import yaml
from sklearn.metrics import roc_auc_score

from kaggle_metric import score as kaggle_score

HORIZON = 168  # hours in one weekly case

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

torch 2.14.1 | device: mps


## 2. Load data

Same loading helpers as `baseline_transformer.ipynb`.

In [2]:
BASE_PATH = Path(".")
KERNEL_PATH = BASE_PATH / "data" / "kernel"
EXTENDED_PATH = BASE_PATH / "data" / "extended"


def load_series(path, date_col="date"):
    df = pd.read_csv(path, parse_dates=[date_col])
    df[date_col] = df[date_col].dt.tz_localize(None)
    return df


def series_arrays(df, date_col="date"):
    value_cols = [c for c in df.columns if c != date_col]
    return df[date_col].to_numpy(), df[value_cols].to_numpy(dtype=float), value_cols


price = load_series(KERNEL_PATH / "Historical_day_ahead_price_2015_2025.csv")
inflow = load_series(KERNEL_PATH / "Historical_inflow_1958_2025.csv")
volume = load_series(KERNEL_PATH / "Historical_volume_2015_2024.csv")
water_value = load_series(KERNEL_PATH / "Synthetic_water_value_2015_2024.csv")
min_flow = load_series(EXTENDED_PATH / "Constraint_min_flow.csv")
min_volume = load_series(EXTENDED_PATH / "Constraint_min_volume.csv")

uc = pd.read_csv(KERNEL_PATH / "Unit_commitment_decisions.csv", parse_dates=["starttime"])
label_cols = [c for c in uc.columns if c.startswith("result_committed")]

price_dates, price_values, _ = series_arrays(price)
inflow_dates, inflow_values, inflow_cols = series_arrays(inflow)
volume_dates, volume_values, volume_cols = series_arrays(volume)
water_value_dates, water_value_values, water_value_cols = series_arrays(water_value)
min_flow_dates, min_flow_values, min_flow_cols = series_arrays(min_flow)
min_volume_dates, min_volume_values, min_volume_cols = series_arrays(min_volume)

N_DYNAMIC = 1 + len(inflow_cols)
N_STATIC = len(volume_cols) + len(water_value_cols) + len(min_flow_cols) + len(min_volume_cols)
print(f"{len(label_cols)} label columns, {len(uc)} labeled weekly cases")
print(f"dynamic features per hour: {N_DYNAMIC} (price + {len(inflow_cols)} inflow columns)")
print(f"static context features: {N_STATIC}")

2352 label columns, 2922 labeled weekly cases
dynamic features per hour: 19 (price + 18 inflow columns)
static context features: 43


## 3. Build sequence tensors

Same as `baseline_transformer.ipynb`: generator order derived programmatically from the generator-major
label columns, with a round-trip assertion guarding the reshape.

In [3]:
def build_sequence_features(starttimes):
    starttimes = pd.to_datetime(starttimes).to_numpy()

    price_pos = np.searchsorted(price_dates, starttimes)
    assert np.all(price_dates[price_pos] == starttimes), "starttime not aligned with price grid"
    idx168 = price_pos[:, None] + np.arange(HORIZON)[None, :]
    price_seq = price_values[idx168]  # (n, 168, 1)

    inflow_pos = np.searchsorted(inflow_dates, starttimes)
    assert np.all(inflow_dates[inflow_pos] == starttimes), "starttime not aligned with inflow grid"
    inflow_idx168 = inflow_pos[:, None] + np.arange(HORIZON)[None, :]
    inflow_seq = inflow_values[inflow_idx168]  # (n, 168, len(inflow_cols))

    dynamic_seq = np.concatenate([price_seq, inflow_seq], axis=-1).astype(np.float32)

    volume_feat = volume_values[np.searchsorted(volume_dates, starttimes)]
    water_value_feat = water_value_values[np.searchsorted(water_value_dates, starttimes)]
    min_flow_feat = min_flow_values[np.searchsorted(min_flow_dates, starttimes)]
    min_volume_feat = min_volume_values[np.searchsorted(min_volume_dates, starttimes)]
    static_ctx = np.hstack(
        [volume_feat, water_value_feat, min_flow_feat, min_volume_feat]
    ).astype(np.float32)

    return dynamic_seq, static_ctx


_pattern = re.compile(r"^result_committed_(.+)_t(\d+)$")
generator_names = []
_seen = set()
for c in label_cols:
    name = _pattern.match(c).group(1)
    if name not in _seen:
        _seen.add(name)
        generator_names.append(name)
n_generators = len(generator_names)
print(f"{n_generators} generators, in column order: {generator_names}")

uc_sorted = uc.sort_values("starttime").reset_index(drop=True)
dynamic_all, static_all = build_sequence_features(uc_sorted["starttime"])

Y_flat = uc_sorted[label_cols].to_numpy(dtype=np.float32)  # (N, 2352), generator-major
N = Y_flat.shape[0]
Y_all = Y_flat.reshape(N, n_generators, HORIZON).transpose(0, 2, 1)  # (N, 168, 14)

Y_roundtrip = Y_all.transpose(0, 2, 1).reshape(N, -1)
assert np.array_equal(Y_roundtrip, Y_flat), "label reshape does not round-trip correctly"

print("dynamic_all:", dynamic_all.shape, "static_all:", static_all.shape, "Y_all:", Y_all.shape)

14 generators, in column order: ['Hogga_G1', 'Tokke_G1', 'Tokke_G2', 'Tokke_G3', 'Tokke_G4', 'Lio_G1', 'Byrte_G1', 'Vinje_G1', 'Vinje_G2', 'Vinje_G3', 'Haukeli_G1', 'Songa_G1', 'Kjela_G1', 'Vesle Kjela_G1']
dynamic_all: (2922, 168, 19) static_all: (2922, 43) Y_all: (2922, 168, 14)


## 4. Time-based train / validation split

Identical to `baseline_transformer.ipynb`.

In [4]:
n_val = int(0.15 * len(uc_sorted))

dynamic_train, dynamic_val = dynamic_all[:-n_val], dynamic_all[-n_val:]
static_train, static_val = static_all[:-n_val], static_all[-n_val:]
Y_train, Y_val = Y_all[:-n_val], Y_all[-n_val:]
run_no_val = uc_sorted["Run No"].iloc[-n_val:].to_numpy()

print(f"train: {dynamic_train.shape[0]} cases, val: {dynamic_val.shape[0]} cases "
      f"({uc_sorted['starttime'].iloc[-n_val].date()} to {uc_sorted['starttime'].iloc[-1].date()})")

train: 2484 cases, val: 438 cases (2021-10-20 to 2022-12-31)


## 5. Normalize

Mean/std fit on the training split only.

In [5]:
dyn_mean = dynamic_train.reshape(-1, dynamic_train.shape[-1]).mean(axis=0)
dyn_std = dynamic_train.reshape(-1, dynamic_train.shape[-1]).std(axis=0) + 1e-6
static_mean = static_train.mean(axis=0)
static_std = static_train.std(axis=0) + 1e-6


def normalize(dynamic_seq, static_ctx):
    return (dynamic_seq - dyn_mean) / dyn_std, (static_ctx - static_mean) / static_std


dynamic_train_n, static_train_n = normalize(dynamic_train, static_train)
dynamic_val_n, static_val_n = normalize(dynamic_val, static_val)

## 6. Dataset / DataLoader

In [6]:
class UCDataset(torch.utils.data.Dataset):
    def __init__(self, dynamic_seq, static_ctx, target):
        self.dynamic_seq = torch.as_tensor(dynamic_seq, dtype=torch.float32)
        self.static_ctx = torch.as_tensor(static_ctx, dtype=torch.float32)
        self.target = torch.as_tensor(target, dtype=torch.float32)

    def __len__(self):
        return len(self.dynamic_seq)

    def __getitem__(self, idx):
        return self.dynamic_seq[idx], self.static_ctx[idx], self.target[idx]


train_dataset = UCDataset(dynamic_train_n, static_train_n, Y_train)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)

dynamic_val_t = torch.as_tensor(dynamic_val_n, dtype=torch.float32).to(device)
static_val_t = torch.as_tensor(static_val_n, dtype=torch.float32).to(device)

print(f"{len(train_dataset)} training sequences, batch size 32 -> {len(train_loader)} batches/epoch")

2484 training sequences, batch size 32 -> 78 batches/epoch


## 7. Models

`UCTransformer` is v1, reproduced identically from `baseline_transformer.ipynb` as the comparison point.
`UCTransformerGenAttrs` is identical except its output head, `GenConditionedHead`, computes each
generator's weight/bias from its static attributes (via a small MLP) plus a learned residual — instead
of a free-standing `Linear(d_model, 14)`.

In [7]:
class UCTransformer(nn.Module):
    def __init__(self, n_dynamic=N_DYNAMIC, n_static=N_STATIC, n_generators=n_generators,
                 horizon=HORIZON, d_model=64, n_heads=4, n_layers=2, d_ff=128, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(n_dynamic, d_model)
        self.ctx_proj = nn.Linear(n_static, d_model)
        self.pos_embedding = nn.Parameter(torch.randn(horizon + 1, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff, dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.output_head = nn.Linear(d_model, n_generators)

    def forward(self, dynamic_seq, static_ctx):
        hourly_tokens = self.input_proj(dynamic_seq)
        ctx_token = self.ctx_proj(static_ctx).unsqueeze(1)
        tokens = torch.cat([ctx_token, hourly_tokens], dim=1)
        tokens = tokens + self.pos_embedding.unsqueeze(0)
        encoded = self.encoder(tokens)
        hourly_encoded = encoded[:, 1:, :]
        return self.output_head(hourly_encoded)


class GenConditionedHead(nn.Module):
    """Per-generator weight/bias come from the generator's static attributes (plus a learned
    residual), instead of a free-standing Linear(d_model, n_generators)."""

    def __init__(self, d_model, gen_features, hidden_dim=32):
        super().__init__()
        self.register_buffer("gen_features", gen_features)  # (n_generators, n_attrs), fixed
        n_generators, n_attrs = gen_features.shape
        self.query_mlp = nn.Sequential(
            nn.Linear(n_attrs, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, d_model)
        )
        self.bias_mlp = nn.Sequential(nn.Linear(n_attrs, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 1))
        self.query_residual = nn.Parameter(torch.zeros(n_generators, d_model))
        self.bias_residual = nn.Parameter(torch.zeros(n_generators))

    def forward(self, hourly_encoded):
        gen_query = self.query_mlp(self.gen_features) + self.query_residual  # (n_generators, d_model)
        gen_bias = self.bias_mlp(self.gen_features).squeeze(-1) + self.bias_residual  # (n_generators,)
        return hourly_encoded @ gen_query.T + gen_bias  # (batch, 168, n_generators)


class UCTransformerGenAttrs(nn.Module):
    def __init__(self, gen_features, n_dynamic=N_DYNAMIC, n_static=N_STATIC, horizon=HORIZON,
                 d_model=64, n_heads=4, n_layers=2, d_ff=128, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(n_dynamic, d_model)
        self.ctx_proj = nn.Linear(n_static, d_model)
        self.pos_embedding = nn.Parameter(torch.randn(horizon + 1, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff, dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.output_head = GenConditionedHead(d_model, gen_features)

    def forward(self, dynamic_seq, static_ctx):
        hourly_tokens = self.input_proj(dynamic_seq)
        ctx_token = self.ctx_proj(static_ctx).unsqueeze(1)
        tokens = torch.cat([ctx_token, hourly_tokens], dim=1)
        tokens = tokens + self.pos_embedding.unsqueeze(0)
        encoded = self.encoder(tokens)
        hourly_encoded = encoded[:, 1:, :]
        return self.output_head(hourly_encoded)

## 8. Generator static attributes

Loaded from `Tokke_Vinje_topology.yaml`, in the same generator order as the label columns, standardized
across the 14 generators.

In [8]:
with open(EXTENDED_PATH / "Tokke_Vinje_topology.yaml") as f:
    topology = yaml.safe_load(f)

gen_attr_names = ["p_min", "p_max", "p_nom", "startcost_const", "stopcost_const", "penstock"]
gen_attr_table = np.array(
    [[topology["model"]["generator"][name][attr] for attr in gen_attr_names] for name in generator_names],
    dtype=np.float32,
)

gen_attr_mean = gen_attr_table.mean(axis=0)
gen_attr_std = gen_attr_table.std(axis=0) + 1e-6
gen_attr_norm = (gen_attr_table - gen_attr_mean) / gen_attr_std
gen_features_tensor = torch.as_tensor(gen_attr_norm, dtype=torch.float32).to(device)

pd.DataFrame(gen_attr_table, index=generator_names, columns=gen_attr_names)

,p_min,p_max,p_nom,startcost_const,stopcost_const,penstock
Hogga_G1,3.000000,17.0,17.0,94.0,94.0,1.0
Tokke_G1,36.900002,107.5,107.5,214.0,214.0,1.0
Tokke_G2,36.900002,107.5,107.5,214.0,214.0,1.0
Tokke_G3,36.900002,107.5,107.5,214.0,214.0,2.0
Tokke_G4,36.900002,107.5,107.5,214.0,214.0,2.0
Lio_G1,14.800000,43.0,43.0,163.0,163.0,1.0
Byrte_G1,6.900000,20.0,20.0,135.0,135.0,1.0
Vinje_G1,34.400002,100.0,100.0,198.0,198.0,1.0
Vinje_G2,34.400002,100.0,100.0,198.0,198.0,2.0
Vinje_G3,34.400002,100.0,100.0,198.0,198.0,3.0


## 9. Train both models

Same `BCEWithLogitsLoss` + `AdamW` + early-stopping-on-validation-AUC setup as `baseline_transformer.ipynb`,
reused for both models via one function.

In [9]:
def predictions_to_submission_frame(proba, run_numbers):
    proba_flat = proba.transpose(0, 2, 1).reshape(proba.shape[0], -1)
    df = pd.DataFrame(proba_flat, columns=label_cols)
    df.insert(0, "Run No", run_numbers)
    return df


Y_val_flat = Y_val.transpose(0, 2, 1).reshape(Y_val.shape[0], -1)
solution_df = pd.DataFrame(Y_val_flat, columns=label_cols)
solution_df.insert(0, "Run No", run_no_val)


def compute_val_auc(model):
    model.eval()
    with torch.no_grad():
        proba = torch.sigmoid(model(dynamic_val_t, static_val_t)).cpu().numpy()
    submission_df = predictions_to_submission_frame(proba, run_no_val)
    return kaggle_score(solution_df, submission_df)


loss_fn = nn.BCEWithLogitsLoss()


def train_with_early_stopping(model, max_epochs=150, patience=15, lr=3e-4, weight_decay=1e-2, log_prefix=""):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_auc, best_epoch, best_state = -1.0, -1, None
    epochs_without_improvement = 0

    start = time.time()
    for epoch in range(max_epochs):
        model.train()
        train_losses = []
        for dyn, stat, tgt in train_loader:
            dyn, stat, tgt = dyn.to(device), stat.to(device), tgt.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(dyn, stat), tgt)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        val_auc = compute_val_auc(model)
        improved = val_auc > best_auc
        if improved:
            best_auc, best_epoch = val_auc, epoch
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epoch % 5 == 0 or improved:
            print(f"{log_prefix}epoch {epoch:3d}  train_loss {np.mean(train_losses):.4f}  "
                  f"val_auc {val_auc:.4f}  best_auc {best_auc:.4f}{' *' if improved else ''}")

        if epochs_without_improvement >= patience:
            print(f"{log_prefix}early stopping at epoch {epoch} (best epoch {best_epoch}, val_auc {best_auc:.4f})")
            break

    model.load_state_dict(best_state)
    print(f"{log_prefix}fit in {time.time() - start:.1f}s")
    return best_auc, best_epoch


torch.manual_seed(42)  # reseed right before each model so both get an identical, isolated init+shuffle
                       # stream -- otherwise whichever model is constructed second inherits whatever
                       # RNG state the first model's init/training left behind, confounding the comparison
model = UCTransformer().to(device)
best_auc_v1, best_epoch_v1 = train_with_early_stopping(model, log_prefix="[v1] ")
print(f"\nv1 best validation micro-average ROC AUC: {best_auc_v1:.4f} (epoch {best_epoch_v1})")

torch.manual_seed(42)
gen_model = UCTransformerGenAttrs(gen_features_tensor).to(device)
best_auc_gen, best_epoch_gen = train_with_early_stopping(gen_model, log_prefix="[gen-attrs] ")
print(f"\ngen-attrs best validation micro-average ROC AUC: {best_auc_gen:.4f} (epoch {best_epoch_gen})")

[v1] epoch   0  train_loss 0.6597  val_auc 0.5529  best_auc 0.5529 *


[v1] epoch   1  train_loss 0.6321  val_auc 0.5767  best_auc 0.5767 *


[v1] epoch   2  train_loss 0.6066  val_auc 0.5958  best_auc 0.5958 *


[v1] epoch   3  train_loss 0.5649  val_auc 0.6064  best_auc 0.6064 *


[v1] epoch   4  train_loss 0.5317  val_auc 0.6197  best_auc 0.6197 *


...

[gen-attrs] epoch  28  train_loss 0.4096  val_auc 0.7129  best_auc 0.7129 *


[gen-attrs] epoch  30  train_loss 0.4042  val_auc 0.7059  best_auc 0.7129


[gen-attrs] epoch  35  train_loss 0.3896  val_auc 0.6991  best_auc 0.7129


[gen-attrs] epoch  40  train_loss 0.3773  val_auc 0.7083  best_auc 0.7129


[gen-attrs] early stopping at epoch 43 (best epoch 28, val_auc 0.7129)
[gen-attrs] fit in 73.6s

gen-attrs best validation micro-average ROC AUC: 0.7129 (epoch 28)


## 10. Per-generator comparison

In [10]:
model.eval()
gen_model.eval()
with torch.no_grad():
    val_proba_v1 = torch.sigmoid(model(dynamic_val_t, static_val_t)).cpu().numpy()
    val_proba_gen = torch.sigmoid(gen_model(dynamic_val_t, static_val_t)).cpu().numpy()

per_gen_auc_v1, per_gen_auc_gen = {}, {}
for i, name in enumerate(generator_names):
    y_true = Y_val[:, :, i].reshape(-1)
    if len(np.unique(y_true)) > 1:
        per_gen_auc_v1[name] = roc_auc_score(y_true, val_proba_v1[:, :, i].reshape(-1))
        per_gen_auc_gen[name] = roc_auc_score(y_true, val_proba_gen[:, :, i].reshape(-1))
    else:
        per_gen_auc_v1[name] = per_gen_auc_gen[name] = float("nan")

comparison = pd.DataFrame({"v1": per_gen_auc_v1, "gen_attrs": per_gen_auc_gen})
comparison["delta"] = comparison["gen_attrs"] - comparison["v1"]
comparison.sort_values("delta")

,v1,gen_attrs,delta
Songa_G1,0.748188,0.673353,-0.074835
Tokke_G1,0.746309,0.674365,-0.071944
Tokke_G2,0.700239,0.634649,-0.065590
Haukeli_G1,0.586071,0.520939,-0.065133
Tokke_G4,0.712785,0.652454,-0.060331
Tokke_G3,0.722651,0.667784,-0.054866
Vesle Kjela_G1,0.824805,0.771371,-0.053435
Kjela_G1,0.761949,0.711960,-0.049988
Hogga_G1,0.720401,0.671326,-0.049075
Lio_G1,0.728957,0.681733,-0.047224


## 11. Conclusion

Conditioning the output head on static generator attributes did **not** improve the overall validation
AUC (gen-attrs ~0.70 vs v1 ~0.71 — see the printed scores in section 9, and the per-generator deltas
above). It helped several generators meaningfully (e.g. `Haukeli_G1`, `Songa_G1`, `Vinje_G3`, `Tokke_G3`)
but hurt others more (notably `Lio_G1`, `Kjela_G1`), for a net wash.

Plausible reason `Lio_G1` regressed: it's the only generator at its plant with no close attribute-twin
among the other 13, so tying its output weights to a function of static attributes (shared via the small
MLP) took away capacity the free-standing `Linear` layer was otherwise free to spend fitting its specific
pattern.

**Recommendation**: don't pursue static-attribute conditioning further as-is. If generator-specific
structure is still worth exploring, a *learned* generator embedding (not derived from attributes) would
isolate whether it's the "shared weights across generators" structure itself that hurts `Lio_G1`,
independent of what the attributes encode. `baseline_transformer.ipynb` keeps using its original v1
model (`Linear(d_model, 14)` head) for the actual submission.